In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
import time

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df = pl.read_parquet("../data/processed/flights_model.parquet")

In [3]:
cols_drop = [
    "AIRLINE_DELAY",
    "WEATHER_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
]

df_model = df.drop(cols_drop)

# remover nulos da target
df_model = df_model.filter(pl.col("ARRIVAL_DELAY").is_not_null())

In [4]:
feature_cols = [
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "AIRLINE",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "SCHEDULED_DEPARTURE",
    "SCHEDULED_TIME",
    "DISTANCE",
    "periodo_dia",
]

target_col = "ARRIVAL_DELAY"

df_model = df_model.select(feature_cols + [target_col])

In [5]:
df_pd = df_model.to_pandas()

X = df_pd[feature_cols]
y = df_pd[target_col]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
categorical_features = [
    "AIRLINE",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "periodo_dia",
]

numeric_features = [
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "SCHEDULED_DEPARTURE",
    "SCHEDULED_TIME",
    "DISTANCE",
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

In [ ]:
# Cache do preprocessador (fit apenas uma vez)
preprocessor.fit(X_train)
print("✓ Preprocessor cached - fit uma única vez")

# Linear Regression Pipeline
start = time.time()
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

lr_pipeline.fit(X_train, y_train)
lr_time = time.time() - start
print(f"⏱ Linear Regression treinou em {lr_time:.2f}s")

In [ ]:
y_pred_lr = lr_pipeline.predict(X_test)

In [ ]:
metrics_lr = {
    "MAE": mean_absolute_error(y_test, y_pred_lr),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_lr)),
    "R2": r2_score(y_test, y_pred_lr),
}

print("Linear Regression:")
print(metrics_lr)

In [ ]:
start = time.time()
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=50,  # Reduzido de 100 para acelerar
        max_depth=8,      # Reduzido de 12 para acelerar
        random_state=42,
        n_jobs=-1,
    )),
])

rf_pipeline.fit(X_train, y_train)
rf_time = time.time() - start
print(f"⏱ Random Forest treinou em {rf_time:.2f}s")

In [ ]:
y_pred_rf = rf_pipeline.predict(X_test)

In [ ]:
# Modelo alternativo mais rápido: GradientBoosting
start = time.time()
gb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
    )),
])

gb_pipeline.fit(X_train, y_train)
gb_time = time.time() - start
print(f"⏱ GradientBoosting treinou em {gb_time:.2f}s")

y_pred_gb = gb_pipeline.predict(X_test)

In [ ]:
metrics_rf = {
    "MAE": mean_absolute_error(y_test, y_pred_rf),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_rf)),
    "R2": r2_score(y_test, y_pred_rf),
}

metrics_gb = {
    "MAE": mean_absolute_error(y_test, y_pred_gb),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_gb)),
    "R2": r2_score(y_test, y_pred_gb),
}

# Comparação
import pandas as pd
comparison = pd.DataFrame({
    "Linear Regression": metrics_lr,
    "Random Forest": metrics_rf,
    "Gradient Boosting": metrics_gb,
}).T

print("\nComparação de Modelos:")
print(comparison)
print(f"\nTempo de Treinamento:")
print(f"  LR: {lr_time:.2f}s | RF: {rf_time:.2f}s | GB: {gb_time:.2f}s")